In [35]:
import pandas as pd
import json
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from collections import defaultdict
import joblib
from owlready2 import *
import torch
import numpy as np
from datasets import Dataset, DatasetDict
import pickle

## PaNET ontology prep work

In [30]:
# Load the ontology
ontology_path='/Users/fdp54928/Documents/PaNET-classifier/data/owlapi.xrdf'
onto=get_ontology(ontology_path).load()

# Run reasoner
with onto:
    sync_reasoner()  # Runs reasoning and updates inferred relationships

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit:/Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/jt/qsn_d43943dbh0h9ngcrb3n80000gq/T/tmp3nrgeeps
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
* Owlready2 * HermiT took 1.0627291202545166 seconds
* Owlready * Equivalenting: PaNET.PaNET01022 wiki.Q133900
* Owlready * Equivalenting: wiki.Q133900 PaNET.PaNET01022
* Owlready * Equivalenting: PaNET.PaNET2014001 PaNET.PaNET2020070
* Owlready * Equivalenting: PaNET.PaNET2020070 PaNET.PaNET2014001

In [31]:
root = onto.search_one(iri='http://purl.org/pan-science/PaNET/PaNET00001')
panet_dict = {}
for cls in root.descendants():
    if cls.iri is not None and len(cls.label) > 0:
        panet_dict[cls.iri] = cls.label[0]
    else:
        print(f"Warning: Class {cls} has no label or IRI.")

In [ ]:
# Save the IRI-to-label dictionary as a json file
# Checkpoint data
with open('/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/iri_to_label_dict.json','w') as f:
    json.dump(panet_dict, f, indent=4)

In [ ]:
# Load the IRI-to-label dictionary if needed
# Checkpoint data

with open('/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/iri_to_label_dict.json') as f:
    panet_dict = json.load(f)

In [32]:
def add_classes_recursively(cls, subclass_map):

    excluded_iris = { 
    'http://www.w3.org/2002/07/owl#Thing',
    'https://www.wikidata.org/wiki/Q133900'
    }

    if cls.iri not in excluded_iris:
        ls = [subclass.iri for subclass in cls.subclasses()]
        subclass_map[cls.iri] = ls
        for subclass in cls.subclasses():
            add_classes_recursively(subclass, subclass_map)
    return subclass_map



root = onto.search_one(iri='http://purl.org/pan-science/PaNET/PaNET00001')
subclass_map = {}

subclass_map = add_classes_recursively(root, subclass_map)

In [37]:
# Save the subclass map as a pkl file
# Checkpoint data

filepath = '/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/'
with open(filepath + 'subclass_map.pkl', 'wb') as f:
    pickle.dump(subclass_map, f)

In [38]:
# Load the subclass map if needed
# Checkpoint data

filepath = '/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/subclass_map.pkl'
subclass_map = joblib.load(filepath)

## Data prep for HGCLR

### The 'raw' json data

In [56]:
# import the data

path = '/Users/fdp54928/Documents/PaNET-classifier/baseline_model/data/'
train_df = pd.read_parquet(path + 'train_set.parquet', engine='fastparquet')
test_df = pd.read_parquet(path + 'test_set.parquet', engine='fastparquet')
val_df = pd.read_parquet(path + 'val_set.parquet', engine='fastparquet')

In [57]:
# Process the datasets

def preprocess_df_text(df):
    """
    :param df: DataFrame, the entire DataFrame
    :return: List[Dict{'token': List[Str], 'label': []}]
    """
    raw_data = list()
    col_names = ['DOI','Title','Abstract']
    for _, row in df.iterrows():
        title = row['Title'].strip().rstrip('.')
        abstract = row['Abstract'].strip()
        line = title + '. ' + abstract
        labels_df = row.drop(columns=col_names)
        labels_ls = list(labels_df[labels_df == 1].index)
        raw_data.append({'token': line.rstrip(), 'label': labels_ls})
    return raw_data


def save_processed_file(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')


def load_processed_file(file_path):
    """
    :param file_path: Str, file path of the processed file
    :return: List[Dict{'token': List[Str], 'label': []}]
    """
    loaded_data = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            # Load each line individually and append to our list
            loaded_data.append(json.loads(line))
    return loaded_data

In [58]:
train_raw = preprocess_df_text(train_df)
test_raw = preprocess_df_text(test_df)
val_raw = preprocess_df_text(val_df)

full_data_raw = train_raw + val_raw + test_raw

In [59]:
# Code to save the raw data
# Checkpoint data

filepath = '/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/'
save_processed_file(train_raw, filepath + 'train_raw.json')
save_processed_file(test_raw, filepath + 'test_raw.json')
save_processed_file(val_raw, filepath + 'val_raw.json')
save_processed_file(full_data_raw, filepath + 'full_data_raw.json')

### The final processed data

In [60]:
def preprocess_data(datapath, tokenizer, binarizer):
    source = []
    labels = []

    with open(datapath, 'r') as f:
        for line in f.readlines():
            line = json.loads(line)
            source.append(tokenizer.encode(line['token'].strip().lower(), truncation=True, max_length=512))
            labels.append(line['label'])
    
    # Use the binarizer directly — consistent with how train/test split was originally encoded
    one_hot_labels = binarizer.transform(labels).tolist()

    
    # --- Create HuggingFace Dataset ---
    dataset = Dataset.from_dict({
        "input_ids": source,          # list of token id lists (variable length)
        "labels": one_hot_labels,     # list of one-hot int lists
    })

    return dataset


In [61]:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
binarizer = joblib.load('/Users/fdp54928/Documents/PaNET-classifier/baseline_model/data/binarizer.pkl')

# panet_dict => mapping from label id to label name
with open('/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/iri_to_label_dict.json') as f:
    panet_dict = json.load(f)

# Load the subclass map
# subclass_map => parent to children mapping
filepath = '/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/subclass_map.pkl'
subclass_map = joblib.load(filepath)

# Process the raw full dataset
full_dataset = preprocess_data('/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/checkpoint_data/full_data_raw.json', tokenizer, binarizer)


value_dict = {i: tokenizer.encode(panet_dict[v].lower(), add_special_tokens=False)
        for i, v in enumerate(binarizer.classes_)}

panet_idx_dict = {v: i for i, v in enumerate(binarizer.classes_)}
hiera = defaultdict(set)

for i, v in enumerate(binarizer.classes_):
    for child in subclass_map[v]:
        if child in panet_idx_dict:
            hiera[i].add(panet_idx_dict[child])


torch.save(value_dict, '/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/processed_data/bert_value_dict.pt')
torch.save(hiera, '/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/processed_data/slot.pt')


train, test, val = [], [], []

train = list(range(len(train_dataset)))
val = list(range(len(train), len(val_dataset)+len(train)))
test = list(range(len(train)+len(val), len(test_dataset)+len(train)+len(val)))

torch.save({'train': train, 'val': val, 'test': test}, '/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/processed_data/split.pt')


full_dataset.save_to_disk('/Users/fdp54928/Documents/PaNET-classifier/HGCLR/data/processed_data')

Saving the dataset (0/1 shards):   0%|          | 0/11502 [00:00<?, ? examples/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Saving the dataset (1/1 shards): 100%|██████████| 11502/11502 [00:00<00:00, 941177.66 examples/s]
